# 10 — Human paired brain ischemia transcript panel (GSE162955)

Six patients, postmortem **infarct core vs matched contralateral tissue**, Affymetrix HTA 2.0 microarray (GPL17586).

This is a whole-brain homogenate signal, **not purified endothelium or pericytes**, and neither PET ligand binding nor BBB transport. Tissue necrosis, cell-mixture changes, agonal/postmortem effects and contralateral disease-related effects confound the estimand.

**Statistics**: within-patient paired differences; 64 exact sign-flip configurations, not a cell-based or independent-sample t-test. Probe-to-gene annotation is audited; multiple probes are aggregated *without choosing the one with the strongest case/control difference*. 

Primary GEO: <https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE162955>


In [ ]:
!pip -q install "numpy>=1.26" "pandas>=2.1" "scipy>=1.12" "statsmodels>=0.14" requests

import io, gzip, os, itertools,hashlib, json, sys
from pathlib import Path
import numpy as np,pandas as pd,requests
ROOT=Path(os.environ.get("IMACS_ROOT","/content/IMACS_GSE162955" if Path("/content").exists() else "./IMACS_GSE162955"))
RAW=ROOT/"raw"; OUT=ROOT/"results"
for x in (RAW,OUT): x.mkdir(parents=True,exist_ok=True)
HEAD={"User-Agent":"IMACS-Stroke-NVU-Research/1.0"}
def dl(url,filename):
 p=RAW/filename
 if not p.is_file() or p.stat().st_size==0:
  with requests.get(url,headers=HEAD,stream=True,timeout=180) as r:
   r.raise_for_status()
   with p.open("wb") as f:
    for chunk in r.iter_content(1<<20):
     if chunk:f.write(chunk)
 return p
def digest(p):
 h=hashlib.sha256()
 with p.open("rb") as f:
  for b in iter(lambda:f.read(1<<20),b""):h.update(b)
 return h.hexdigest()
SERIES="https://ftp.ncbi.nlm.nih.gov/geo/series/GSE162nnn/GSE162955/matrix/GSE162955_series_matrix.txt.gz"
ANN="https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL17nnn/GPL17586/annot/GPL17586.annot.gz"
p=dl(SERIES,"GSE162955_series_matrix.txt.gz")
q=dl(ANN,"GPL17586.annot.gz")
pd.DataFrame([{"source_url":u,"file":f.name,"sha256":digest(f),"bytes":f.stat().st_size}
              for u,f in [(SERIES,p),(ANN,q)]]).to_csv(OUT/"10_GEO_input_checksums.csv",index=False)
print(p.stat().st_size,q.stat().st_size)


## Verify paired patient mapping from GEO GSM titles

Six *known* pairs, one core and one contralateral for each patient. Never match by array position alone.

In [ ]:
pairs={
"16":("GSM4968878","GSM4968879"),
"22":("GSM4968880","GSM4968881"),
"32":("GSM4968882","GSM4968883"),
"33":("GSM4968884","GSM4968885"),
"35":("GSM4968886","GSM4968887"),
"36":("GSM4968888","GSM4968889")}
with gzip.open(p,"rt",encoding="utf-8",errors="replace") as f:
 data=[]; meta=[]
 inside=False
 for line in f:
  if line.startswith("!series_matrix_table_begin"):inside=True;continue
  if line.startswith("!series_matrix_table_end"):break
  if inside:data.append(line)
  else:meta.append(line)
assert data,"Could not locate series matrix table"
M=pd.read_csv(io.StringIO("".join(data)),sep="\t",low_memory=False)
print(M.shape,list(M.columns[:5]))
probe_col=M.columns[0]
M[probe_col]=M[probe_col].astype(str)
M=M.set_index(probe_col)
required=[gsm for pair in pairs.values() for gsm in pair]
assert set(required).issubset(M.columns),f"Missing required paired GSM columns: {set(required)-set(M.columns)}"
M=M[required].apply(pd.to_numeric,errors="coerce")
pd.DataFrame({"sample":M.columns,"n_nonmissing":M.notna().sum().values,
             "min":M.min().values,"max":M.max().values}).to_csv(OUT/"10_microarray_sample_quality.csv",index=False)
print(M.describe().loc[["min","25%","50%","75%","max"]])
# Normalized Affymetrix intensity must be already log2; stop if not.
median=float(M.median().median())
if median>25 or median<1:
 raise ValueError(f"Microarray intensities do not resemble pre-log2 data (median {median}); do not log2 twice or test untransformed blindly.")


## Extract GPL17586 official platform mapping

Ambiguous multi-symbol mappings are excluded from frozen candidate inference; the original platform source and count of unmatched probes are preserved.

In [ ]:
with gzip.open(q,"rt",encoding="utf-8",errors="replace") as f:
 lines=[]; found=False
 for ln in f:
  if ln.startswith("!platform_table_begin"):found=True;continue
  if ln.startswith("!platform_table_end"):break
  if found:lines.append(ln)
assert lines,"GEO GPL .annot file lacked an annotation table"
A=pd.read_csv(io.StringIO("".join(lines)),sep="\t",low_memory=False)
print("ANNOT COLS",list(A.columns),"shape",A.shape)
idcol=next((c for c in A.columns if c.lower() in ("id","id_ref","probe_id","transcript_cluster_id")),None)
assert idcol is not None, "No authoritative probe ID column"
symcol=next((c for c in A.columns if c.lower().replace("_"," ").strip() in ("gene symbol","gene symbols","symbol","gene name")),None)
if symcol is None:
 symcol=next((c for c in A.columns if "symbol" in c.lower()),None)
if symcol is None:
 symcol=next((c for c in A.columns if "gene_assignment" in c.lower()),None)
assert symcol is not None,"No verified gene-symbol annotation: stop rather than map by guessing"
print("Mapping:",idcol,symcol)
mapdf=A[[idcol,symcol]].copy()
mapdf.columns=["probe","symbol"]
mapdf["probe"]=mapdf["probe"].astype(str)
mapdf["symbol"]=mapdf["symbol"].astype(str).str.strip()
if "assignment" in symcol.lower():
 mapdf["symbol"]=mapdf["symbol"].str.split("//",regex=False).str[1].str.strip()
mapdf=mapdf[~mapdf["symbol"].isna()]
# Keep only one unambiguous HGNC-style symbol, exclude compound / uncertain.
mapdf=mapdf[mapdf["symbol"].str.fullmatch(r"[A-Za-z][A-Za-z0-9_.-]*",na=False)]
mapdf=mapdf.drop_duplicates(subset=["probe","symbol"])
ambig=mapdf.groupby("probe").symbol.nunique()
mapdf=mapdf[~mapdf.probe.isin(ambig[ambig>1].index)]
mapdf.to_csv(OUT/"10_GPL17586_usable_probe_symbol_mapping.csv",index=False)
print("usable mappings",len(mapdf),"microarray overlap",mapdf.probe.isin(M.index).sum())


## Frozen candidate panel and six-pair exact sign flips

If multiple probes map uniquely to a gene, take the median per patient and tissue **before** comparing groups. Do not cherry-pick the probe giving the largest log fold change.

In [ ]:
catalog="https://raw.githubusercontent.com/Touf00/retinal-nvu-computational-analysis/imacs-stroke-pet-screen/IMACS_Stroke_PET/config/candidate_catalog.csv"
cat=pd.read_csv(catalog).drop_duplicates("gene")
genes=set(cat.gene.astype(str).str.upper())
mapdf["gene_up"]=mapdf.symbol.str.upper()
targets=mapdf[mapdf.gene_up.isin(genes)]
v=M.reindex(targets.probe)
v["gene_up"]=targets.gene_up.values
gene_matrix=v.groupby("gene_up")[required].median()
mapping_n=targets.groupby("gene_up").probe.nunique()
rows=[]; paired=[]
for gene in cat.gene:
 key=str(gene).upper()
 if key not in gene_matrix.index:
  rows.append({"gene":gene,"status":"NO_UNAMBIGUOUS_MATCH","n_probes":0})
  continue
 vals=gene_matrix.loc[key]
 ds=[]
 for patient,(cl,ic) in pairs.items():
  delta=vals[ic]-vals[cl]
  ds.append(float(delta))
  paired.append({"gene":gene,"patient_id":patient,"contralateral_intensity":vals[cl],
                 "infarct_core_intensity":vals[ic],"paired_core_minus_contra":delta})
 ds=np.asarray(ds,dtype=float)
 if not np.isfinite(ds).all():
  rows.append({"gene":gene,"status":"MISSING_PATIENT_INTENSITY","n_probes":int(mapping_n[key])})
  continue
 obs=abs(float(ds.mean()))
 null=np.array([abs(np.mean(ds*np.array(signs,dtype=int)))
                for signs in itertools.product([-1,1],repeat=6)])
 pval=float(np.mean(null>=obs-1e-12))
 rows.append({"gene":gene,"status":"TESTABLE","n_probes":int(mapping_n[key]),
              "mean_paired_core_minus_contra_log2_intensity":float(ds.mean()),
              "median_paired_core_minus_contra_log2_intensity":float(np.median(ds)),
              "paired_exact_signflip_p_two_sided":pval,
              "n_positive_pairs":int((ds>0).sum()),"n_negative_pairs":int((ds<0).sum()),
              "n_pairs":6})
out=pd.DataFrame(rows)
# BH is over the frozen 29-gene panel's tested subset only; no genome-wide significance.
from statsmodels.stats.multitest import multipletests
idx=out.status.eq("TESTABLE")
out.loc[idx,"BH_q_frozen_panel"]=multipletests(out.loc[idx,"paired_exact_signflip_p_two_sided"],method="fdr_bh")[1]
out.to_csv(OUT/"10_human_paired_core_vs_contralateral_candidate_screen.csv",index=False)
pd.DataFrame(paired).to_csv(OUT/"10_human_per_patient_candidate_intensities.csv",index=False)
display(out)


## /TEAMRED review checklist

This experiment is **postmortem bulk tissue**, not purified vessels or RNA from surviving penumbral endothelial cells. Loss of expression in infarct core may reflect cellular loss rather than target downregulation. Ischemic core is neither peri-infarct angiogenesis nor an IV-administered accessible target. Paired contralateral cortex may have diaschisis/pathology and is not a normal volunteer control. Probe platform is old (HTA 2.0) and the mapping must be audited carefully. Do not meta-analyze these microarray effects with mouse single-cell log2CPM.

Only use this dataset to question or provisionally support the **human presence and injury sensitivity** of a transcript; protein/luminal localization is an independent requirement.


In [ ]:
(OUT/"10_run_manifest.json").write_text(json.dumps({
 "accession":"GSE162955","organism":"Homo sapiens","platform":"GPL17586",
 "n_paired_donors":6,"comparisons":"autopsy infarct core vs matched contralateral",
 "statistic":"exact two-sided six-pair signflip, 64 configurations",
 "normalization":"author processed microarray log2 intensity",
 "celltype_resolution":"none; whole tissue",
 "target_protein_or_PET_validation":False
},indent=2))
print("HUMAN PAIRED WHOLE-TISSUE ANALYSIS COMPLETE")
